# Company research data and download workflow

Keep the three source workbooks and `Research_Data.csv.gz` beside this notebook. The gzip file contains **one CSV**, compressed for download. No transport folders or reconstruction scripts are needed. Read it directly below; do not expand it to open the entire file in a spreadsheet.

Inputs: `Company_List_for_Interns.xlsx`, `CapitalIQ_560.xlsx`, `WRDS_560.xlsx`. Downloaded data includes LSEG financials/company/segments, provided financial data, FF49 context, current facets, existing comparable candidates and source documents. Working-process logs and before/after review files are omitted.


In [ ]:
from pathlib import Path
import csv, json, gzip, os
import pandas as pd
csv.field_size_limit(2**31-1)
ROOT=Path.cwd()
DATA=ROOT/'Research_Data.csv.gz'
assert DATA.exists(), 'Download Research_Data.csv.gz from this repository release and put it beside the notebook.'
def records(dataset):
    with gzip.open(DATA,'rt',encoding='utf-8-sig',newline='') as f:
        for row in csv.DictReader(f):
            if row['dataset']==dataset:
                item=json.loads(row['record_json'])
                if dataset=='Source_Documents':item['Source Text']=row['text']
                yield item
def table(dataset):
    return pd.DataFrame.from_records(records(dataset))
contents=next(records('Contents'))
pd.Series(contents['Dataset Counts'],name='rows').sort_index()

## Read company data, facets and sources

In [ ]:
facets=table('Company_Facets')
wendy_core=table('Wendys_Company_Facets')
wendy_segments=table('Wendys_Segment_Facets')
comps=table('Comparable_Candidates')
print('Company facets:',len(facets),'Wendy pilot:',len(wendy_core),len(wendy_segments),'existing pairs:',len(comps))
display(wendy_core[['Facet','Value Text','Source Links','Document Provenance']].head(10))

In [ ]:
annual=table('LSEG_annual_five_years')
display(annual.head())
# Complete stored source text; URLs and dates remain in the source records.
source=next(records('Source_Documents'),None)
if source:
    print(source.get('Source Link'))
    print(source.get('Source Text','')[:5000])

## Original inputs

In [ ]:
import zipfile, xml.etree.ElementTree as ET
for name in ['Company_List_for_Interns.xlsx','CapitalIQ_560.xlsx','WRDS_560.xlsx']:
    with zipfile.ZipFile(ROOT/name) as archive:
        workbook=ET.fromstring(archive.read('xl/workbook.xml'))
        sheets=[node.attrib['name'] for node in workbook.iter() if node.tag.endswith('}sheet')]
        print(name,sheets)
# To read Excel rows, install openpyxl and then use:
# input_data=pd.read_excel(ROOT/'CapitalIQ_560.xlsx',sheet_name='Business Description')


## Download code

These functions show how source data is fetched. They are defined but not run automatically. Viewing the saved CSV needs no keys. WRDS and LSEG refreshes require an authorized account and locally configured credentials; no secret is embedded in this notebook. SEC EDGAR needs a contact User-Agent.


In [ ]:
import requests, time

def fetch_wrds(table_name='comp/company',limit=100,offset=0):
    token=os.environ.get('WRDS_API_TOKEN')
    if not token: raise RuntimeError('Set WRDS_API_TOKEN in your local environment.')
    if table_name not in {'comp/company','comp/g_company'}:raise ValueError('Review table selection first.')
    response=requests.get(f'https://wrds-api.wharton.upenn.edu/data/{table_name}/',
        params={'limit':limit,'offset':offset},headers={'Authorization':'Token '+token},timeout=60)
    response.raise_for_status()
    return response.json()

def fetch_sec_submissions(cik):
    contact=os.environ.get('SEC_USER_AGENT')
    if not contact:raise RuntimeError('Set SEC_USER_AGENT to your name and contact email.')
    url=f'https://data.sec.gov/submissions/CIK{int(cik):010d}.json'
    response=requests.get(url,headers={'User-Agent':contact},timeout=45)
    response.raise_for_status();time.sleep(.25)
    return response.json()

def fetch_filing_text(url):
    from html.parser import HTMLParser
    class Text(HTMLParser):
        def __init__(self):super().__init__();self.parts=[];self.skip=0
        def handle_starttag(self,tag,attrs):
            if tag in {'script','style'}:self.skip+=1
        def handle_endtag(self,tag):
            if tag in {'script','style'}:self.skip=max(0,self.skip-1)
        def handle_data(self,text):
            if not self.skip:self.parts.append(text)
    contact=os.environ.get('SEC_USER_AGENT')
    if not contact:raise RuntimeError('Configure SEC_USER_AGENT locally.')
    response=requests.get(url,headers={'User-Agent':contact},timeout=60)
    response.raise_for_status();parser=Text();parser.feed(response.text);time.sleep(.25)
    return '\n'.join(t.strip() for t in parser.parts if t.strip())

def fetch_lseg(instruments,fields,parameters=None):
    import lseg.data as ld
    # Uses your authorized local SDK configuration / running Workspace session.
    ld.open_session()
    try:return ld.get_data(universe=instruments,fields=fields,parameters=parameters or {},header_type=ld.HeaderType.NAME)
    finally:ld.close_session()

# Examples, intentionally not executed:
# wrds_page=fetch_wrds()
# sec_index=fetch_sec_submissions(30697)
# five_years=fetch_lseg(['WEN.O'],['TR.TotalRevenue','TR.EBITDA'],{'SDate':'0','EDate':'-4','Period':'FY','Curn':'USD'})


## Extend the CSV with downloaded records

Keep provider field names, currency, units, observation date, filing/publication date and exact source URL. A stored text excerpt must be copied from the source, not silently rewritten. Financial data for pseudo-private targets remains diagnostic under the website-only matching design. Existing candidate scores are provisional and were not recomputed from the expanded Wendy pilot.


In [ ]:
def save_new_snapshot(dataframe, dataset, output='New_Download.csv'):
    # Explicit output prevents overwriting the delivered master accidentally.
    rows=[]
    for item in dataframe.to_dict('records'):
        rows.append({'dataset':dataset,'record_json':json.dumps(item,ensure_ascii=False,default=str)})
    pd.DataFrame(rows).to_csv(ROOT/output,index=False)

# Minimal structural checks of the delivered research profile.
assert len(wendy_core)==30 and len(wendy_segments)==42
assert wendy_core.Facet.is_unique
assert wendy_core.loc[wendy_core.Facet=='Primary Products','Value Text'].notna().all()
assert len(comps)==5490
print('Data loaded and basic structural checks passed. No external API was called.')